# Information-or-Placebo pilot: FULL 225-request run - fresh runtime, built for Save & Run All

**Run it with: File -> Save Version -> Save & Run All (Commit).** Do not run cells by hand.
Settings before saving: Accelerator **GPU T4 x2** · Internet **On** · Add Input -> dataset
**info-placebo-pilot-bundle** (bundle **v5**, exactly one copy attached).

Frozen protocol (no change): Qwen/Qwen3-4B @ 3101254bbe4169895668a0e7653c3fd1f313576e · FP16 · cuda:0 ·
Kaggle T4 · temperature 0 · do_sample False · num_beams 1 · enable_thinking False · max_new_tokens 768 ·
batch size 1 · stateless single user message, no system message, no history · frozen datasets, prompts,
conditions, false-hint assignments, execution order (seed 2026), parser and scorer.
225 requests = 40 x L0, V, L_false_near, L, A + 25 x L_false_far.

Every cell refuses to run unless the previous one completed, so a failed check can never be followed by
experimental requests. Outputs (kept as the saved version's Output):
`results/full_768/` · `checkpoints/checkpoint_025.zip ... checkpoint_225.zip` · **`results_full_768_FINAL.zip`**.
No repair rates, CCA, comparisons or tests are computed.

In [ ]:
# CELL 1 - GPU / environment check (fresh runtime required; Kaggle T4; no CPU fallback)
import hashlib, json, sys
from pathlib import Path
hits = [m.parent for m in Path('/kaggle/input').rglob('manifest.json')
        if json.loads(m.read_text()).get('bundle_id') == 'info-placebo-pilot-bundle']
assert len(hits) == 1, f'STOP: attach exactly one info-placebo-pilot-bundle dataset (found {hits})'
BUNDLE_IN = hits[0]
_m = json.loads((BUNDLE_IN / 'manifest.json').read_text())
assert _m.get('bundle_version') == 5, f"STOP: bundle v5 required, found v{_m.get('bundle_version')}"
for _f in ('scripts/kaggle_runner.py', 'scripts/kaggle_full_run.py'):
    assert hashlib.sha256((BUNDLE_IN / _f).read_bytes()).hexdigest() == _m['files'][_f], f'STOP: {_f} hash mismatch'
sys.path.insert(0, str(BUNDLE_IN / 'scripts'))
import kaggle_full_run as kf
GPU = kf.cell1_environment()

In [ ]:
# CELL 2 - install / verify the audited package versions (transformers 5.19.0, accelerate 1.15.0)
VERSIONS = kf.cell2_packages()

In [ ]:
# CELL 3 - stage the bundle in /kaggle/working/pilot and verify every file hash (frozen data included)
MANIFEST = kf.cell3_stage()

In [ ]:
# CELL 4 - verify the pinned revision on the Hub, then load the exact Qwen3-4B checkpoint (FP16, cuda:0)
BACKEND = kf.cell4_load()

In [ ]:
# CELL 5 - verify FP16, cuda:0, non-thinking chat template (single user message) and greedy generation
TEMPLATE = kf.cell5_verify(BACKEND)

In [ ]:
# CELL 6 - small non-experimental 2+2 canary (stored only in results/diagnostics/)
CANARY = kf.cell6_canary(BACKEND)

In [ ]:
# CELL 7 - full pre-flight validation for the 225-request experiment (STOP here = nothing is sent)
CTX = kf.cell7_preflight(BACKEND, GPU, VERSIONS, MANIFEST)

In [ ]:
# CELL 8 - the exact deterministic 225 requests -> /kaggle/working/results/full_768/
# fsync after every request; checkpoints/checkpoint_025.zip ... checkpoint_225.zip ("CHECKPOINT READY: N/225")
kf.cell8_run(BACKEND, CTX)

In [ ]:
# CELL 9 - parse and score with the frozen parser + scorer (row counts only are shown)
SCORED = kf.cell9_score()

In [ ]:
# CELL 10 - final completeness / integrity diagnostics (no analysis)
DIAG = kf.cell10_diagnostics(CTX, SCORED)

In [ ]:
# CELL 11 - archive the complete run and verify it
FINAL = kf.cell11_final_zip()
from IPython.display import HTML, display
display(HTML("<div style='border:6px solid #c00;padding:18px;font-size:22px;font-weight:bold;color:#c00'>"
             "RUN COMPLETE<br>DOWNLOAD FROM SAVED VERSION OUTPUT:<br>results_full_768_FINAL.zip<br>"
             f"<span style='font-size:14px'>{FINAL['bytes']} bytes &middot; SHA-256 {FINAL['sha256']}</span></div>"))